# EC2 &middot; Classificação de incidentes

**Minicurso "Inteligência Artificial aplicada à Resposta a Incidentes" &middot; SBSeg 2026**

Este *notebook* acompanha a Seção 1.6 do capítulo e opera sobre a saída do
`01-anonimizacao.ipynb`. Ele reproduz, em escala didática, os quatro cenários
experimentais que o SecLINC e o AutoClass-LFI avaliam sobre dados reais.

## O que você vai responder

| Etapa | Pergunta |
|---|---|
| 1 | Quantos rótulos distintos a **categorização livre** inventa? |
| 2 | Quanto a **taxonomia estruturada** reduz essa dispersão? |
| 3 | Quanto **um único exemplo por categoria** acrescenta? |
| 4 | A **heurística iterativa** (PHP) muda alguma decisão? |
| 5 | Onde a **matriz de confusão** concentra os erros, e de quem é a culpa? |

**Saída:** `saida/incidentes_classificados.csv`, entrada do *notebook* 03.

> **Métrica por categoria, nunca acurácia global.** Na base de *tickets* reais
> do minicurso, dois terços dos registros são CAT5: um classificador que
> respondesse sempre CAT5 marcaria ~65% de acurácia sem nenhuma capacidade
> discriminativa. Nossa base sintética reproduz esse desbalanceamento de
> propósito.

## Configuração: escolhendo o provedor

Três provedores, uma variável. Leia com atenção o que cada um significa.

| Provedor | O que é | Quando usar |
|---|---|---|
| `simulado` | Classificador determinístico por regras e similaridade, escrito para esta atividade. **Não é um LLM.** | Sem GPU, sem rede, sem credencial. É o padrão. |
| `ollama` | Modelo de pesos abertos rodando em `localhost`. | Quando quiser materializar o argumento de privacidade: o dado não sai da máquina. |
| `openai` | Qualquer endpoint compatível com *chat completions*. | Só sobre dados já pseudonimizados: isso envia o texto a um terceiro. |

**Sobre o provedor `simulado`:** ele existe para que a mecânica do *pipeline*,
das métricas e da comparação entre estratégias seja exercitável em qualquer
máquina. Os números que ele produz **não** devem ser lidos como desempenho de
modelo de linguagem. Ele reproduz deliberadamente dois fenômenos documentados
no capítulo: a dispersão de rótulos na categorização livre, e o ganho do
*one-shot*, este último por um mecanismo real (vizinho mais próximo sobre os
exemplos fornecidos).

In [ ]:
import csv, hashlib, json, os, re, unicodedata
from collections import Counter, defaultdict
from pathlib import Path

PROVEDOR = "simulado"          # "simulado" | "ollama" | "openai"
MODELO   = "qwen3:14b"         # usado por "ollama" e "openai"
TEMPERATURA = 0.0              # classificação: sempre 0

RAIZ = Path.cwd()
if not (RAIZ / "dados").is_dir():
    RAIZ = RAIZ.parent
DADOS, SAIDA = RAIZ / "dados", RAIZ / "saida"
SAIDA.mkdir(exist_ok=True)

TAXONOMIA = {
    "CAT1":  "Comprometimento de Conta",
    "CAT2":  "Malware",
    "CAT3":  "Negacao de Servico (DoS/DDoS)",
    "CAT4":  "Exfiltracao ou Vazamento",
    "CAT5":  "Exploracao de Vulnerabilidade",
    "CAT6":  "Abuso Interno",
    "CAT7":  "Engenharia Social",
    "CAT8":  "Incidente Fisico",
    "CAT9":  "Alteracao Nao Autorizada",
    "CAT10": "Uso Indevido de Recursos",
    "CAT11": "Problema de Terceiro",
    "CAT12": "Tentativa de Intrusao",
}
PRIORIDADE = {"CAT1": 5, "CAT2": 5, "CAT4": 5, "CAT5": 5, "CAT6": 5,
              "CAT3": 4, "CAT8": 4, "CAT11": 4,
              "CAT7": 3, "CAT9": 3, "CAT12": 3, "CAT10": 2}

# --- Entrada: saída do notebook 01 ---------------------------------------
entrada = SAIDA / "tickets_pseudonimizados.csv"
if not entrada.exists():
    entrada = DADOS / "tickets_sinteticos.csv"
    print("[aviso] saida/tickets_pseudonimizados.csv nao encontrado.")
    print("        Usando os tickets sinteticos NAO pseudonimizados.")
    print("        Rode o 01-anonimizacao.ipynb para o pipeline completo.")

with open(entrada, encoding="utf-8") as f:
    TICKETS = list(csv.DictReader(f))

print(f"provedor: {PROVEDOR}   |   {len(TICKETS)} tickets de {entrada.name}")
print("distribuicao real:",
      dict(sorted(Counter(t["categoria"] for t in TICKETS).items(),
                  key=lambda kv: -kv[1])))

## Os provedores

In [ ]:
# ---------------------------------------------------------------------
# Provedor 1: simulado (determinístico, offline)
# ---------------------------------------------------------------------
# Classificador por léxico ponderado. Cada categoria tem termos que a
# indicam; a pontuação é a soma dos pesos dos termos presentes. Quando o
# prompt traz exemplos rotulados, soma-se a similaridade de cosseno com
# cada exemplo, ponderada, ao escore da categoria do exemplo. É por isso
# que o one-shot melhora de verdade aqui: o mecanismo existe.

# ATENÇÃO: os termos abaixo são vocabulário *genérico* de segurança, do tipo
# que um modelo pré-treinado já conhece. Deliberadamente NÃO incluímos frases
# copiadas dos tickets desta base: isso inflaria o zero-shot e apagaria a
# lição da etapa 3. Repare também nas sobreposições propositais entre CAT3,
# CAT5 e CAT12: elas reproduzem a ambiguidade real da tarefa.
LEXICO = {
    "CAT1":  {"comprometimento de conta": 3, "credencial": 2, "conta": 1,
              "sessoes": 2, "senha": 2, "segundo fator": 2},
    "CAT2":  {"malware": 4, "ransomware": 4, "edr": 3, "antivirus": 3,
              "binario": 2, "infeccao": 3, "cifrag": 3, "resgate": 3,
              "persistencia": 2},
    "CAT3":  {"ddos": 4, "negacao de servico": 4, "denial of service": 4,
              "amplificacao": 3, "volumetrico": 3, "gbps": 3, "indisponib": 2,
              "satur": 2, "blackhole": 2, "refletor": 2},
    "CAT4":  {"vazamento": 4, "exfiltracao": 4, "exposicao": 3, "dump": 2,
              "lgpd": 1, "anpd": 1, "credenciais": 1},
    "CAT5":  {"vulnerabilidade": 4, "cve": 3, "exploracao": 3, "cvss": 2,
              "patch": 2, "desatualiz": 2, "injection": 3, "falha": 2,
              "scanner": 1, "openvas": 2, "atualizar": 1},
    "CAT6":  {"abuso interno": 4, "etica": 2, "indevido": 2, "funcionari": 2},
    "CAT7":  {"phishing": 4, "engenharia social": 4, "fraude": 3, "golpe": 3,
              "remetente": 2, "pix": 2, "link": 1},
    "CAT8":  {"fisic": 3, "infiltracao": 3, "rompimento": 3, "energia": 2,
              "datacenter": 2, "rack": 2, "incendio": 3},
    "CAT9":  {"alteracao": 3, "nao autorizada": 2, "modificacao": 3,
              "defacement": 4, "mudanca": 2},
    "CAT10": {"mineracao": 4, "criptomoeda": 4, "uso indevido": 3,
              "recursos": 1, "xmrig": 3},
    "CAT11": {"fornecedor": 4, "terceiro": 3, "contratada": 3, "parceiro": 1},
    "CAT12": {"intrusao": 4, "forca bruta": 4, "varredura": 3, "scan": 3,
              "failed password": 3, "tentativa": 2, "honeypot": 2,
              "reconhecimento": 2, "bloqueou": 1},
}


def _norm(texto):
    t = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in t if not unicodedata.combining(c))


def _sacola(texto):
    return Counter(re.findall(r"[a-z0-9]{4,}", _norm(texto)))


def _cosseno(a, b):
    comuns = set(a) & set(b)
    if not comuns:
        return 0.0
    num = sum(a[t] * b[t] for t in comuns)
    den = (sum(v * v for v in a.values()) ** 0.5) * \
          (sum(v * v for v in b.values()) ** 0.5)
    return num / den if den else 0.0


def _escores(texto, exemplos=None):
    t = _norm(texto)
    escore = {c: 0.0 for c in TAXONOMIA}
    for cat, termos in LEXICO.items():
        escore[cat] = sum(peso for termo, peso in termos.items() if termo in t)
    if exemplos:
        sacola = _sacola(texto)
        for ex in exemplos:
            escore[ex["categoria"]] += 12.0 * _cosseno(sacola, _sacola(ex["texto"]))
    return escore


def _simulado_taxonomia(texto, exemplos=None):
    escore = _escores(texto, exemplos)
    melhor = max(escore.values())
    if melhor == 0:
        return "CAT5"                                  # viés da classe majoritária
    empatados = [c for c, v in escore.items() if v == melhor]
    return max(empatados, key=lambda c: PRIORIDADE[c])  # regra do prompt


# Variantes de rótulo livre, para reproduzir a dispersão semântica que
# Severo et al. (2025) documentam no cenário SHCL.
_VARIANTES = {
    "CAT3":  ["DDoS Attack", "Distributed Denial of Service (DDoS)",
              "Ataque de Negacao de Servico Distribuida", "DDoS volumetrico"],
    "CAT5":  ["Vulnerability Exploitation", "Exploracao de Vulnerabilidade",
              "Servico mal configurado exposto", "Vulnerabilidade critica"],
    "CAT12": ["Intrusion Attempt", "Tentativa de acesso nao autorizado",
              "Brute Force / Scan", "Reconhecimento"],
    "CAT2":  ["Malware Infection", "Infeccao por codigo malicioso",
              "Ransomware"],
    "CAT7":  ["Phishing", "Social Engineering", "Fraude / Engenharia Social"],
}


def _simulado_livre(texto):
    cat = _simulado_taxonomia(texto)
    opcoes = _VARIANTES.get(cat, [TAXONOMIA[cat]])
    idx = int(hashlib.sha256(_norm(texto).encode()).hexdigest(), 16) % len(opcoes)
    return opcoes[idx]

In [ ]:
# ---------------------------------------------------------------------
# Provedores 2 e 3: modelo local (Ollama) e API compatível com OpenAI
# ---------------------------------------------------------------------
def _chamar_ollama(prompt):
    import requests
    r = requests.post("http://localhost:11434/api/chat", timeout=180, json={
        "model": MODELO, "stream": False,
        "options": {"temperature": TEMPERATURA},
        "messages": [{"role": "user", "content": prompt}]})
    r.raise_for_status()
    return r.json()["message"]["content"]


def _chamar_openai(prompt):
    import requests
    base = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com")
    chave = os.environ["OPENAI_API_KEY"]
    r = requests.post(f"{base}/v1/chat/completions", timeout=180,
                      headers={"Authorization": f"Bearer {chave}"},
                      json={"model": MODELO, "temperature": TEMPERATURA,
                            "messages": [{"role": "user", "content": prompt}]})
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def classificar(texto, modo="taxonomia", exemplos=None, dica=None):
    """modo: 'livre' | 'taxonomia'. exemplos habilita one-shot; dica, o PHP."""
    if PROVEDOR == "simulado":
        if modo == "livre":
            return _simulado_livre(texto)
        # O simulado é estável: a dica do PHP confirma a resposta anterior.
        return _simulado_taxonomia(texto, exemplos)

    prompt = montar_prompt(texto, modo, exemplos, dica)
    bruto = (_chamar_ollama if PROVEDOR == "ollama" else _chamar_openai)(prompt)
    return bruto.strip() if modo == "livre" else extrair_categoria(bruto)


def montar_prompt(texto, modo, exemplos=None, dica=None):
    if modo == "livre":
        return ("Voce e um classificador de incidentes de ciberseguranca.\n"
                "Analise o relato abaixo e responda APENAS com a categoria "
                f"do incidente.\n\nINCIDENTE:\n{texto}")
    lista = "\n".join(f"{c}  {n}" for c, n in TAXONOMIA.items())
    partes = ["Voce e um classificador de incidentes de ciberseguranca.",
              "Classifique o relato abaixo em EXATAMENTE UMA das categorias:",
              "", lista, "",
              "Responda APENAS com o codigo da categoria (ex.: CAT5).",
              "Se houver mais de uma categoria plausivel, escolha a de maior "
              "prioridade."]
    if exemplos:
        partes += ["", "EXEMPLOS ROTULADOS:"]
        for ex in exemplos:
            partes += ["--- Exemplo ---",
                       f"INCIDENTE: {ex['texto'][:400]}",
                       f"CATEGORIA: {ex['categoria']}"]
    if dica:
        partes += ["", f"DICA: uma analise anterior deste mesmo incidente "
                       f"sugeriu a categoria {dica}. Reavalie criticamente. "
                       f"Se a dica estiver correta, confirme-a; se estiver "
                       f"errada, corrija-a."]
    partes += ["", "INCIDENTE:", texto]
    return "\n".join(partes)


def extrair_categoria(bruto):
    """LLMs raramente respondem só o código, apesar da instrução."""
    m = re.search(r"\bCAT(\d{1,2})\b", bruto.upper())
    if m and f"CAT{m.group(1)}" in TAXONOMIA:
        return f"CAT{m.group(1)}"
    alvo = _norm(bruto)
    for cod, nome in TAXONOMIA.items():                # tenta pelo nome
        if _norm(nome).split(" (")[0] in alvo:
            return cod
    return "INDEFINIDO"


def tabela(linhas, cabecalho):
    try:
        import pandas as pd
        return pd.DataFrame(linhas, columns=cabecalho)
    except ImportError:
        larg = [max(len(str(c)), *(len(str(l[i])) for l in linhas or [cabecalho]))
                for i, c in enumerate(cabecalho)]
        print("  ".join(str(c).ljust(w) for c, w in zip(cabecalho, larg)))
        print("  ".join("-" * w for w in larg))
        for l in linhas:
            print("  ".join(str(v).ljust(w) for v, w in zip(l, larg)))
        return None


print("provedores prontos. ativo:", PROVEDOR)

## Etapa 1 &middot; Categorização livre: medindo a dispersão

Sem lista de categorias, o modelo produz rótulos heterogêneos. Severo et al.
(2025) documentam ataques de negação de serviço aparecendo com pelo menos
quatro denominações distintas no mesmo conjunto. Vamos contar.

In [ ]:
livres = [classificar(t["texto"], modo="livre") for t in TICKETS]
distintos = Counter(livres)

print(f"{len(TICKETS)} tickets  ->  {len(distintos)} rotulos distintos")
print(f"(a taxonomia tem {len(TAXONOMIA)} categorias, e nem todas ocorrem "
      f"na base: {len(set(t['categoria'] for t in TICKETS))})\n")
tabela(sorted(distintos.items(), key=lambda kv: -kv[1]),
       ["rotulo produzido", "ocorrencias"])

### Leitura da etapa 1

O problema não é o modelo estar *errado*: cada rótulo isolado é defensável. O
problema é que a saída **não é agregável**. Não dá para contar incidentes por
categoria, comparar meses, nem recuperar o *playbook* correspondente, porque a
chave de recuperação muda a cada execução.

É esse o argumento a favor de vocabulário controlado, e ele é operacional,
não estético.

## Etapa 2 &middot; Orientada por taxonomia, *zero-shot*

In [ ]:
for t in TICKETS:
    t["pred_zs"] = classificar(t["texto"], modo="taxonomia")

acertos = sum(t["pred_zs"] == t["categoria"] for t in TICKETS)
print(f"acuracia global (zero-shot): {acertos}/{len(TICKETS)} "
      f"= {100*acertos/len(TICKETS):.1f}%")
print("rotulos distintos:", len({t['pred_zs'] for t in TICKETS}),
      "(era", len(distintos), "na categorizacao livre)")

## Etapa 3 &middot; *One-shot*, com exclusão dos exemplos

Jesus Filho et al. (2025) mostram que um único exemplo rotulado por categoria
pode elevar o F1 médio de ~46% para mais de 90%. A interpretação dos autores é
que o exemplo **não ensina o conceito**, que o modelo já conhece: ele comunica
*como aquela organização específica* desenha a fronteira entre categorias.

**Cuidado metodológico obrigatório:** os *tickets* usados como exemplo precisam
sair do conjunto de avaliação. Sem isso, o número medido é contaminado.

In [ ]:
# Regra de seleção: o **medoide** de cada categoria, isto é, o ticket mais
# parecido com os demais da mesma categoria. Escolher o exemplo mais
# representativo, e não o primeiro que aparece, é uma prática estabelecida em
# few-shot e evita que a atividade dependa da ordem do arquivo.
#
# Só categorias com 2 ou mais tickets entram: com uma única ocorrência, usar
# o ticket como exemplo o retiraria da avaliação e a categoria ficaria sem
# suporte. Ficar sem exemplo para 5 das 12 categorias é, aliás, o cenário
# realista de uma base desbalanceada.
por_categoria = defaultdict(list)
for t in TICKETS:
    por_categoria[t["categoria"]].append(t)

exemplos = []
for cat, itens in por_categoria.items():
    if len(itens) < 2:
        continue
    sacolas = [_sacola(i["texto"]) for i in itens]
    medoide = max(range(len(itens)), key=lambda a: sum(
        _cosseno(sacolas[a], sacolas[b]) for b in range(len(itens)) if b != a))
    exemplos.append({"texto": itens[medoide]["texto"], "categoria": cat,
                     "id": itens[medoide]["id"]})

# Exclusão do conjunto de avaliação. É o passo que quase todo tutorial
# esquece, e que infla a acurácia reportada.
ids_exemplo = {e["id"] for e in exemplos}
AVALIACAO = [t for t in TICKETS if t["id"] not in ids_exemplo]

sem_exemplo = sorted(set(TAXONOMIA) - {e["categoria"] for e in exemplos},
                     key=lambda c: int(c[3:]))
print(f"{len(exemplos)} exemplos (um por categoria com >= 2 tickets): "
      f"{sorted((e['categoria'], e['id']) for e in exemplos)}")
print(f"categorias SEM exemplo: {sem_exemplo}")
print(f"conjunto de avaliacao: {len(AVALIACAO)} tickets "
      f"(de {len(TICKETS)}; {len(ids_exemplo)} excluidos)")

for t in AVALIACAO:
    t["pred_1s"] = classificar(t["texto"], modo="taxonomia", exemplos=exemplos)

a_zs = sum(t["pred_zs"] == t["categoria"] for t in AVALIACAO)
a_1s = sum(t["pred_1s"] == t["categoria"] for t in AVALIACAO)
print(f"\nsobre o MESMO conjunto de avaliacao ({len(AVALIACAO)} tickets):")
print(f"  zero-shot : {a_zs:2d} acertos = {100*a_zs/len(AVALIACAO):.1f}%")
print(f"  one-shot  : {a_1s:2d} acertos = {100*a_1s/len(AVALIACAO):.1f}%")

> **Não leia esses percentuais como resultado.** Com 17 *tickets* de avaliação,
> um acerto a mais move a acurácia em quase 6 pontos, e o classificador é um
> simulador por regras, não um LLM. O que este *notebook* demonstra é a
> **mecânica** do *few-shot* e o cuidado de exclusão. Para magnitude, veja a
> Tabela 1.9 do capítulo, medida sobre 216 *tickets* reais.
>
> **Olhe _quais_ erros somem, não quantos.** Com o provedor `simulado`, o único
> erro corrigido pelo *one-shot* é o `T017`, a notificação de servidor NTP com
> `monlist` habilitado. No *zero-shot* ele vai para CAT3, e com razão aparente:
> o texto fala em "amplificação" e em "fator superior a 200x". Só que a
> convenção desta organização registra a *notificação de serviço mal
> configurado* como CAT5, porque o incidente é a vulnerabilidade, não o ataque
> que ela viabiliza. Nenhum modelo adivinha essa convenção a partir do
> conhecimento geral. O que o exemplo rotulado faz não é ensinar o conceito de
> DDoS, que o modelo já conhece: é comunicar **onde esta organização traça a
> fronteira**. O exemplo escolhido para CAT5 (`T001`, resolvedor DNS recursivo
> aberto) carrega exatamente essa fronteira, e por isso transfere. É a leitura
> de Jesus Filho et al. (2025), em miniatura.

## Etapa 4 &middot; *Progressive-Hint Prompting*

O PHP reenvia a pergunta acompanhada de uma dica derivada da resposta anterior,
até que respostas sucessivas se estabilizem. No SecLINC, a estabilização é
medida por ROUGE com limiar 0,9; aqui, por simplicidade, paramos quando duas
rodadas consecutivas devolvem o mesmo código.

In [ ]:
def classificar_php(texto, exemplos=None, max_dicas=3):
    """Devolve (categoria, n_rodadas, historico)."""
    historico = [classificar(texto, "taxonomia", exemplos)]
    for _ in range(max_dicas):
        nova = classificar(texto, "taxonomia", exemplos, dica=historico[-1])
        historico.append(nova)
        if nova == historico[-2]:                 # estabilizou
            break
    return historico[-1], len(historico), historico


mudou = 0
for t in AVALIACAO:
    t["pred_php"], rodadas, hist = classificar_php(t["texto"], exemplos)
    if len(set(hist)) > 1:
        mudou += 1
        print(f"  {t['id']}: {' -> '.join(hist)}  (referencia: {t['categoria']})")

a_php = sum(t["pred_php"] == t["categoria"] for t in AVALIACAO)
print(f"\nPHP: {a_php}/{len(AVALIACAO)} = {100*a_php/len(AVALIACAO):.1f}%")
print(f"tickets em que a dica mudou a decisao: {mudou}")
if PROVEDOR == "simulado":
    print("\n[nota] o provedor simulado e deterministico e estavel por "
          "construcao:\n       a dica sempre confirma a resposta anterior e "
          "o ciclo para na 2a rodada.\n       Troque para 'ollama' para ver o "
          "PHP operando de verdade.")

## Etapa 5 &middot; Métricas por categoria e matriz de confusão

Acurácia global sobre base desbalanceada é uma métrica enganosa. Reportamos
precisão, *recall* e F1 **por categoria**, mais o F1 macro, que pondera
igualmente todas as classes.

In [ ]:
def relatorio(verdadeiros, preditos, titulo):
    """Precisão, recall e F1 por classe, sem depender de scikit-learn."""
    classes = sorted(set(verdadeiros) | set(preditos),
                     key=lambda c: int(c[3:]) if c.startswith("CAT") else 99)
    vp = Counter(); fp = Counter(); fn = Counter(); suporte = Counter(verdadeiros)
    for v, p in zip(verdadeiros, preditos):
        if v == p:
            vp[v] += 1
        else:
            fp[p] += 1; fn[v] += 1

    linhas, f1s = [], []
    for c in classes:
        prec = vp[c] / (vp[c] + fp[c]) if vp[c] + fp[c] else 0.0
        rec  = vp[c] / (vp[c] + fn[c]) if vp[c] + fn[c] else 0.0
        f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        if suporte[c]:
            f1s.append(f1)
        linhas.append((c, f"{prec:.2f}", f"{rec:.2f}", f"{f1:.2f}", suporte[c]))

    acc = sum(v == p for v, p in zip(verdadeiros, preditos)) / len(verdadeiros)
    print(f"\n===== {titulo} =====")
    print(f"acuracia global: {acc:.3f}   |   F1 macro: "
          f"{sum(f1s)/len(f1s):.3f}   (classes com suporte: {len(f1s)})")
    tabela(linhas, ["classe", "precisao", "recall", "f1", "suporte"])
    return acc, sum(f1s) / len(f1s)


y = [t["categoria"] for t in AVALIACAO]
for coluna, nome in [("pred_zs", "zero-shot"), ("pred_1s", "one-shot"),
                     ("pred_php", "PHP")]:
    relatorio(y, [t[coluna] for t in AVALIACAO], nome)

# Conferência opcional contra a implementação de referência
try:
    from sklearn.metrics import classification_report
    print("\n--- conferencia com scikit-learn (one-shot) ---")
    print(classification_report(y, [t["pred_1s"] for t in AVALIACAO],
                                digits=2, zero_division=0))
except ImportError:
    print("\n[scikit-learn ausente: conferencia pulada]")

In [ ]:
def matriz_confusao(verdadeiros, preditos):
    classes = sorted(set(verdadeiros) | set(preditos),
                     key=lambda c: int(c[3:]) if c.startswith("CAT") else 99)
    m = defaultdict(Counter)
    for v, p in zip(verdadeiros, preditos):
        m[v][p] += 1
    print("linhas = referencia humana | colunas = predicao\n")
    print(" " * 7 + "".join(c.rjust(7) for c in classes))
    for v in classes:
        if not sum(m[v].values()):
            continue
        print(v.rjust(6) + "".join(
            (str(m[v][p]) if m[v][p] else ".").rjust(7) for p in classes))
    erros = [(t["id"], t["categoria"], t["pred_1s"])
             for t in AVALIACAO if t["pred_1s"] != t["categoria"]]
    if erros:
        print("\nerros do one-shot:")
        tabela(erros, ["ticket", "referencia", "predito"])


matriz_confusao(y, [t["pred_1s"] for t in AVALIACAO])

### As três perguntas de discussão

1. **Quantos rótulos distintos a categorização livre produziu?** Compare com as
   12 categorias da taxonomia. Esse número é o argumento mais direto a favor de
   vocabulário controlado.

2. **Onde a matriz de confusão concentra os erros?** Procure a confusão entre
   **CAT5** (exploração de vulnerabilidade) e **CAT12** (tentativa de intrusão).
   Elas são conceitualmente adjacentes: um *port scan* seguido de tentativa de
   exploração pode ser lido das duas formas, e é por isso que Severo et al.
   registram que as falhas se concentram em incidentes com múltiplas categorias
   plausíveis. Pergunta que decorre: **o erro é do modelo ou da taxonomia?**

3. **Qual é a diferença entre o modelo local e o comercial, e ela compensa?**
   A resposta não é técnica, é de política de dados. Almeida et al. (2025)
   medem ~60% de acurácia para modelos abertos de 8B a 20B contra mais de 90%
   para comerciais. Em troca: o dado não sai da organização, o custo é
   previsível, o modelo é versionado e auditável, e não há dependência de
   jurisdição estrangeira. A arquitetura defensável é em dois níveis, com
   triagem local de todo o volume e escalonamento seletivo.

## Etapa 6 &middot; Gerar a entrada do EC3

In [ ]:
# Confiança didática: distância entre o melhor e o segundo melhor escore.
# Em um LLM real, use log-probabilidades, voto majoritário entre execuções,
# ou a estabilização do PHP como proxy.
def confianca(texto, exemplos=None):
    e = sorted(_escores(texto, exemplos).values(), reverse=True)
    if e[0] == 0:
        return 0.0
    return round(min(1.0, (e[0] - e[1]) / e[0]), 2)


destino = SAIDA / "incidentes_classificados.csv"
with open(destino, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, quoting=csv.QUOTE_ALL, fieldnames=[
        "id", "id_incidente", "texto", "categoria_referencia",
        "categoria_predita", "confianca", "revisar"])
    w.writeheader()
    for t in TICKETS:
        pred = t.get("pred_1s", t["pred_zs"])
        conf = confianca(t["texto"], exemplos) if PROVEDOR == "simulado" else ""
        # Regra de encaminhamento para revisão humana, conforme o capítulo:
        # baixa confiança OU categoria de prioridade máxima.
        revisar = (conf != "" and conf < 0.35) or PRIORIDADE.get(pred, 0) == 5
        w.writerow({"id": t["id"], "id_incidente": t["id_incidente"],
                    "texto": t["texto"],
                    "categoria_referencia": t["categoria"],
                    "categoria_predita": pred, "confianca": conf,
                    "revisar": "sim" if revisar else "nao"})

with open(destino, encoding="utf-8") as f:
    linhas = list(csv.DictReader(f))
n_rev = sum(l["revisar"] == "sim" for l in linhas)
print(f"{len(linhas)} incidentes classificados -> {destino.relative_to(RAIZ)}")
print(f"encaminhados a revisao humana: {n_rev} ({100*n_rev/len(linhas):.0f}%)")
print("\nExatamente esse encaminhamento e o que o Modulo PoP do GT-LFI")
print("implementa: a classificacao automatica e uma proposta submetida a")
print("julgamento, nao uma decisao.")

## Discussão e limites

**Quando confiar na classificação automática.** Para *roteamento* e
*priorização inicial*, quando três condições se verificam simultaneamente: usa-se
taxonomia estruturada; usa-se heurística iterativa ou ao menos um exemplo por
categoria; e o incidente pertence a uma categoria bem representada na base. Fora
disso, e especialmente em categorias de interpretação subjetiva, a saída é
sugestão.

**O que exige revisão humana.** Quatro situações: incidentes com múltiplas
categorias plausíveis; categorias de prioridade máxima, nas quais o custo do
erro é alto; relatos genéricos ou incompletos (rode o *ticket* `T024` e veja);
e qualquer caso em que execuções sucessivas divirjam.

**O teto prático.** Sobre 265 *tickets* reais rotulados independentemente por
dois analistas, Jesus Filho et al. (2025) registram divergência entre eles em
~18,5% dos casos. Esse número é o teto de qualquer classificador avaliado
contra rótulo humano: um modelo que concordasse 100% com o analista A
discordaria do analista B em cerca de um a cada cinco.

**Como isso alimenta o EC3.** A categoria atribuída é a chave de recuperação do
*playbook*. Um erro aqui propaga-se como recomendação de resposta inadequada
adiante, e o erro é silencioso, porque a saída seguinte é sintaticamente válida
e internamente coerente. Por isso o Módulo PoP posiciona a validação humana
**entre** a classificação e a geração, e não apenas ao final.

## Exercícios

1. Instale o Ollama, baixe um modelo (`ollama pull qwen3:14b`) e troque
   `PROVEDOR = "ollama"`. Compare as métricas. O que muda na matriz de confusão?
2. Rode a etapa 4 três vezes com `TEMPERATURA = 0.8` e um modelo real. Quantos
   *tickets* mudam de categoria entre execuções? Esse é o problema de
   variabilidade que o capítulo discute.
3. Troque a taxonomia de 12 categorias do NIST pela do CERT.br, com 6
   (DoS, Fraude, Intrusão, Varredura, Web, Outros). A acurácia sobe. Isso
   significa que a taxonomia é melhor, ou que a tarefa ficou mais fácil?
4. Implante voto majoritário: rode a classificação três vezes e fique com a
   moda. Em quantos casos o voto difere da primeira execução?

---

**Próximo passo:** abra `03-playbooks.ipynb`, que consome
`saida/incidentes_classificados.csv`.